<a href="https://colab.research.google.com/github/hewittpeterson/nba-simulator/blob/main/nba_simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random, pandas as pd
from IPython.display import clear_output


class Team:
    def __init__(self, name, offense, defense, conference, division):
        self.name = name
        self.offense = offense
        self.defense = defense
        self.conference = conference
        self.division = division
        self.wins = 0
        self.losses = 0


def simulate_game(home, away, home_score=0, away_score=0, is_ot=False):
    if not is_ot:
        home_score = 0
        away_score = 0

    home_adv = 3
    dampener = 0.25
    home_prob = 50 + ((home.offense - away.defense) * dampener) + home_adv
    away_prob = 50 + ((away.offense - home.defense) * dampener)

    possessions = random.randint(10, 15) if is_ot else random.randint(95, 115)

    for i in range(possessions):
        if random.randint(0, 100) < home_prob:
            home_score += random.choice([1, 2, 2, 2, 2, 2, 3, 3])
        if random.randint(0, 100) < away_prob:
            away_score += random.choice([1, 2, 2, 2, 2, 2, 3, 3])

    status = "OT" if is_ot else "Final"
    if home_score > away_score:
        home.wins += 1
        away.losses += 1
        return f"{away.name} {away_score} - {home.name} {home_score} ({home.name} Win - {status}) "
    elif away_score > home_score:
        away.wins += 1
        home.losses += 1
        return f"{away.name} {away_score} - {home.name} {home_score} ({away.name} Win - {status}) "
    else:
        return simulate_game(home, away, home_score, away_score, is_ot=True)


def generate_schedule(team_list):
    matchup_pool = []
    team_game_counts = {team.name: 0 for team in team_list}

    for i in range(len(team_list)):
        for j in range(i + 1, len(team_list)):
            t1, t2 = team_list[i], team_list[j]
            if t1.conference != t2.conference: count = 2
            elif t1.division == t2.division: count = 4
            else: count = 3

            for _ in range(count):
                matchup_pool.append((t1, t2))
                team_game_counts[t1.name] += 1
                team_game_counts[t2.name] += 1

    while any(count < 82 for count in team_game_counts.values()):
        needing = [t for t in team_list if team_game_counts[t.name] < 82]
        if len(needing) < 2: break

        t1, t2 = needing[0], needing[1]
        matchup_pool.append((t1, t2))
        team_game_counts[t1.name] += 1
        team_game_counts[t2.name] += 1

    random.shuffle(matchup_pool)
    return matchup_pool


def show_standings(team_list):
    standings = []
    for t in team_list:
        total_games = t.wins + t.losses
        win_pct = t.wins / total_games if total_games > 0 else 0
        standings.append({
            "Team": t.name,
            "Conf": t.conference,
            "Wins": t.wins,
            "Losses": t.losses,
            "Win %": round(win_pct, 3)
        })

    df = pd.DataFrame(standings)

    east_df = df[df['Conf'] == 'East'].sort_values(by=["Win %", "Wins"], ascending=False).reset_index(drop=True)
    west_df = df[df['Conf'] == 'West'].sort_values(by=["Win %", "Wins"], ascending=False).reset_index(drop=True)

    print("\n=== EASTERN CONFERENCE ===")
    display(east_df)
    print("\n=== WESTERN CONFERENCE ===")
    display(west_df)


def run_interactive_season(team_list, user_team_name):
    user_team = next((t for t in team_list if user_team_name.lower() in t.name.lower()), None)
    if not user_team:
        print("Team not found!")
        return pd.DataFrame()

    schedule = generate_schedule(team_list)
    results_log = []
    simulating_rest = False

    for i, (home, away) in enumerate(schedule):
        is_user_game = (home.name == user_team.name or away.name == user_team.name)

        if is_user_game and not simulating_rest:
            print(f"\n--- UPCOMING GAME: {away.name} @ {home.name} ---")
            print("Options: [P] Play Game | [S] Sim Rest of Season")
            choice = input("What would you like to do? ").strip().lower()

            if choice == 's' or choice == 'S':
                simulating_rest = True
                print(">>> Fast-forwarding to the end of the season...")
                result_str = simulate_game(home, away)
            else:
                strat = input("Choose Strategy: [1] Focus Offense, [2] Focus Defense, [3] Balanced: ")
                if strat == "1":
                    user_team.offense += 5
                    user_team.defense -= 2
                    print(f"Strategy set: Aggressive Offense!")
                elif strat == "2":
                    user_team.defense += 5
                    user_team.offense -= 2
                    print(f"Strategy set: Lockdown Defense!")

                result_str = simulate_game(home, away)
                if f"{user_team.name} Win" in result_str:
                    print(f"🏆 The {user_team.name} won!")
                    print(f"RESULT: {result_str}")
                else:
                    print(f"😞 Tough loss for the {user_team.name}.")
                    print(f"RESULT: {result_str}")

                if strat == "1":
                    user_team.offense -= 5
                    user_team.defense += 2
                elif strat == "2":
                    user_team.defense -= 5
                    user_team.offense += 2
        else:
            result_str = simulate_game(home, away)
        results_log.append({"Home": home.name, "Away": away.name, "Result": result_str})

    print("\n--- Season Complete! ---")
    show_standings(team_list)
    return pd.DataFrame(results_log)


def search_team_history(team_name, results_df):
    team_games = results_df[results_df['Home'].str.contains(team_name, case=False) | results_df['Away'].str.contains(team_name, case=False)]
    if team_games.empty:
        print(f"No games found for '{team_name}'. Make sure the name matches exactly!")
    else:
        print(f"=== Game History for {team_name} ===")
        display(team_games)



def simulate_series(team1, team2, games_needed=4):
    t1_wins, t2_wins = 0, 0
    while t1_wins < games_needed and t2_wins < games_needed:

        is_t1_home = t1_wins + t2_wins in [0, 1, 4, 6]
        result = simulate_game(team1 if is_t1_home else team2, team2 if is_t1_home else team1)

        if f"{team1.name} Win" in result:
            t1_wins += 1
        else:
            t2_wins += 1

    winner = team1 if t1_wins > t2_wins else team2
    print(f"SERIES RESULT: {team1.name} ({t1_wins}) vs {team2.name} ({t2_wins}) -> {winner.name} advance!")
    return winner

def run_playoffs(team_list):
    print("\n--- STARTING THE NBA PLAYOFFS ---")

    def get_top_8(conf_name):
        conf_teams = [t for t in team_list if t.conference == conf_name]
        return sorted(conf_teams, key=lambda t: (t.wins / (t.wins + t.losses)), reverse=True)[:8]

    east_seeds = get_top_8("East")
    west_seeds = get_top_8("West")

    def run_conference_bracket(seeds, name):
        print(f"\n--- {name.upper()} CONFERENCE BRACKET ---")
        # Round 1: 1v8, 2v7, 3v6, 4v5
        r1_winners = [
            simulate_series(seeds[0], seeds[7]),
            simulate_series(seeds[3], seeds[4]),
            simulate_series(seeds[1], seeds[6]),
            simulate_series(seeds[2], seeds[5])
        ]
        # Conference Semis
        print(f"\n--- {name} Semifinals ---")
        r2_winners = [
            simulate_series(r1_winners[0], r1_winners[1]),
            simulate_series(r1_winners[2], r1_winners[3])
        ]
        # Conference Finals
        print(f"\n--- {name} Finals ---")
        return simulate_series(r2_winners[0], r2_winners[1])

    east_champ = run_conference_bracket(east_seeds, "Eastern")
    west_champ = run_conference_bracket(west_seeds, "Western")

    print("\n🏆🏆🏆 THE NBA FINALS 🏆🏆🏆")
    champion = simulate_series(east_champ, west_champ)
    print(f"\n🎉 THE {champion.name.upper()} ARE THE NBA CHAMPIONS! 🎉")


# NBA Team List
teams = [

    # --- Eastern Conference ---

    # Atlantic Division
    Team("Boston Celtics", 92, 89, "East", "Atlantic"),
    Team("Brooklyn Nets", 78, 75, "East", "Atlantic"),
    Team("New York Knicks", 89, 88, "East", "Atlantic"),
    Team("Philadelphia 76ers", 87, 85, "East", "Atlantic"),
    Team("Toronto Raptors", 82, 80, "East", "Atlantic"),

    # Central Division
    Team("Chicago Bulls", 80, 78, "East", "Central"),
    Team("Cleveland Cavaliers", 88, 86, "East", "Central"),
    Team("Detroit Pistons", 81, 79, "East", "Central"),
    Team("Indiana Pacers", 86, 76, "East", "Central"),
    Team("Milwaukee Bucks", 85, 82, "East", "Central"),

    # Southeast Division
    Team("Atlanta Hawks", 84, 80, "East", "Southeast"),
    Team("Charlotte Hornets", 79, 77, "East", "Southeast"),
    Team("Miami Heat", 83, 87, "East", "Southeast"),
    Team("Orlando Magic", 82, 88, "East", "Southeast"),
    Team("Washington Wizards", 76, 74, "East", "Southeast"),

    # --- Western Conference ---

    # Northwest Division
    Team("Denver Nuggets", 91, 85, "West", "Northwest"),
    Team("Minnesota Timberwolves", 88, 91, "West", "Northwest"),
    Team("Oklahoma City Thunder", 89, 94, "West", "Northwest"),
    Team("Portland Trail Blazers", 78, 76, "West", "Northwest"),
    Team("Utah Jazz", 79, 75, "West", "Northwest"),

    # Pacific Division
    Team("Golden State Warriors", 88, 83, "West", "Pacific"),
    Team("LA Clippers", 86, 85, "West", "Pacific"),
    Team("Los Angeles Lakers", 89, 84, "West", "Pacific"),
    Team("Phoenix Suns", 90, 82, "West", "Pacific"),
    Team("Sacramento Kings", 85, 81, "West", "Pacific"),

    # Southwest Division
    Team("Dallas Mavericks", 90, 80, "West", "Southwest"),
    Team("Houston Rockets", 84, 86, "West", "Southwest"),
    Team("Memphis Grizzlies", 83, 85, "West", "Southwest"),
    Team("New Orleans Pelicans", 85, 83, "West", "Southwest"),
    Team("San Antonio Spurs", 87, 88, "West", "Southwest")
]


In [ ]:
user_choice = input("Enter the team you want to control: ")
season_results_df = run_interactive_season(teams, user_choice)
display(season_results_df.head())

Enter the team you want to control: jazz

--- UPCOMING GAME: Utah Jazz @ Minnesota Timberwolves ---
Options: [P] Play Game | [S] Sim Rest of Season
What would you like to do? p
Choose Strategy: [1] Focus Offense, [2] Focus Defense, [3] Balanced: 1
Strategy set: Aggressive Offense!
😞 Tough loss for the Utah Jazz.
RESULT: Utah Jazz 105 - Minnesota Timberwolves 108 (Minnesota Timberwolves Win - Final) 

--- UPCOMING GAME: Sacramento Kings @ Utah Jazz ---
Options: [P] Play Game | [S] Sim Rest of Season
What would you like to do? p
Choose Strategy: [1] Focus Offense, [2] Focus Defense, [3] Balanced: 1
Strategy set: Aggressive Offense!
🏆 The Utah Jazz won!
RESULT: Sacramento Kings 114 - Utah Jazz 129 (Utah Jazz Win - Final) 

--- UPCOMING GAME: Golden State Warriors @ Utah Jazz ---
Options: [P] Play Game | [S] Sim Rest of Season
What would you like to do? 1
Choose Strategy: [1] Focus Offense, [2] Focus Defense, [3] Balanced: 1
Strategy set: Aggressive Offense!
🏆 The Utah Jazz won!
RESULT: Go

,Team,Conf,Wins,Losses,Win %
0,Boston Celtics,East,68,14,0.829
1,Cleveland Cavaliers,East,59,23,0.720
2,New York Knicks,East,58,24,0.707
3,Philadelphia 76ers,East,50,32,0.610
4,Milwaukee Bucks,East,49,33,0.598
5,Orlando Magic,East,43,39,0.524
6,Chicago Bulls,East,41,41,0.500
7,Indiana Pacers,East,40,42,0.488
8,Miami Heat,East,40,42,0.488
9,Toronto Raptors,East,39,43,0.476



=== WESTERN CONFERENCE ===


,Team,Conf,Wins,Losses,Win %
0,Denver Nuggets,West,57,25,0.695
1,Oklahoma City Thunder,West,56,26,0.683
2,Minnesota Timberwolves,West,52,30,0.634
3,Golden State Warriors,West,45,37,0.549
4,Los Angeles Lakers,West,41,41,0.500
5,Portland Trail Blazers,West,38,44,0.463
6,San Antonio Spurs,West,38,44,0.463
7,LA Clippers,West,36,46,0.439
8,Houston Rockets,West,36,46,0.439
9,New Orleans Pelicans,West,35,47,0.427


,Home,Away,Result
0,Miami Heat,Phoenix Suns,Phoenix Suns 103 - Miami Heat 94 (Phoenix Suns...
1,Denver Nuggets,Memphis Grizzlies,Memphis Grizzlies 108 - Denver Nuggets 115 (De...
2,Washington Wizards,Houston Rockets,Houston Rockets 112 - Washington Wizards 104 (...
3,Brooklyn Nets,Indiana Pacers,Indiana Pacers 129 - Brooklyn Nets 108 (Indian...
4,Portland Trail Blazers,LA Clippers,LA Clippers 103 - Portland Trail Blazers 104 (...


In [ ]:
while True:
  search_name = input("Enter 'exit' or enter a team to search: ")
  if search_name.lower() == 'exit':
    print("Exiting search.")
    break
  clear_output(wait = True)
  search_team_history(search_name, season_results_df)

=== Game History for kings ===


,Home,Away,Result
9,Utah Jazz,Sacramento Kings,Sacramento Kings 114 - Utah Jazz 129 (Utah Jaz...
14,Golden State Warriors,Sacramento Kings,Sacramento Kings 126 - Golden State Warriors 1...
20,New York Knicks,Sacramento Kings,Sacramento Kings 86 - New York Knicks 99 (New ...
61,Golden State Warriors,Sacramento Kings,Sacramento Kings 134 - Golden State Warriors 1...
90,Sacramento Kings,Dallas Mavericks,Dallas Mavericks 116 - Sacramento Kings 131 (S...
...,...,...,...
1109,Denver Nuggets,Sacramento Kings,Sacramento Kings 144 - Denver Nuggets 145 (Den...
1112,Sacramento Kings,New Orleans Pelicans,New Orleans Pelicans 118 - Sacramento Kings 13...
1115,Toronto Raptors,Sacramento Kings,Sacramento Kings 129 - Toronto Raptors 136 (To...
1126,LA Clippers,Sacramento Kings,Sacramento Kings 95 - LA Clippers 109 (LA Clip...


KeyboardInterrupt: Interrupted by user

In [ ]:
run_playoffs(teams)


--- STARTING THE NBA PLAYOFFS ---

--- EASTERN CONFERENCE BRACKET ---
SERIES RESULT: Boston Celtics (4) vs Orlando Magic (2) -> Boston Celtics advances!
SERIES RESULT: Toronto Raptors (3) vs Philadelphia 76ers (4) -> Philadelphia 76ers advances!
SERIES RESULT: New York Knicks (3) vs Milwaukee Bucks (4) -> Milwaukee Bucks advances!
SERIES RESULT: Cleveland Cavaliers (2) vs Atlanta Hawks (4) -> Atlanta Hawks advances!

--- Eastern Semifinals ---
SERIES RESULT: Boston Celtics (4) vs Philadelphia 76ers (1) -> Boston Celtics advances!
SERIES RESULT: Milwaukee Bucks (4) vs Atlanta Hawks (0) -> Milwaukee Bucks advances!

--- Eastern Finals ---
SERIES RESULT: Boston Celtics (4) vs Milwaukee Bucks (1) -> Boston Celtics advances!

--- WESTERN CONFERENCE BRACKET ---
SERIES RESULT: Minnesota Timberwolves (2) vs Sacramento Kings (4) -> Sacramento Kings advances!
SERIES RESULT: LA Clippers (2) vs Phoenix Suns (4) -> Phoenix Suns advances!
SERIES RESULT: Denver Nuggets (2) vs Golden State Warriors (